# 01 · Embedding geometry

This notebook reproduces, on real CelebA face embeddings, the geometric phenomena that the rest of the pipeline depends on:

1. **Concentration of measure**: in high dimensions, random vectors are nearly orthogonal. Above-zero similarity is statistical evidence, not coincidence.
2. **Anisotropy**: untrained or weakly-aligned encoders produce embeddings that cluster in a narrow cone, which collapses discriminative power.
3. **Why SigLIP-2 retrieves better than CLIP**: its sigmoid loss yields a more uniform distribution on the unit hypersphere, and that uniformity translates directly into Recall@K.

These are the same phenomena introduced in Weeks 2 and 3 of the course. Here we measure them on the actual data we use for retrieval.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / "src"))

from poi.data import CelebADataset
from poi.embeddings.factory import build_encoder
from poi.utils.config import EmbeddingConfig

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110

## Setup: a small sample of CelebA

We don't need all 200k faces for this. A few thousand is enough to show the distributional differences clearly.

In [ ]:
DATA_DIR = Path("../data/celeba/img_align_celeba")
N_SAMPLE = 5000
RNG = np.random.default_rng(42)

dataset = CelebADataset(images_dir=DATA_DIR)
all_paths = dataset.list_image_paths()
sample_paths = list(RNG.choice(all_paths, size=N_SAMPLE, replace=False))
print(f"Sampling {len(sample_paths)} faces from {len(all_paths)} total")

## The thought experiment first: random vectors

Before running real encoders, let's reproduce the experiment from Week 2: take random unit vectors in increasing dimensions and look at the distribution of pairwise cosine similarities.

In [ ]:
def random_unit_vectors(n: int, d: int, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    v = rng.standard_normal(size=(n, d)).astype(np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def pairwise_cosine_sample(vectors: np.ndarray, n_pairs: int = 50_000) -> np.ndarray:
    """Random sample of pairwise cosine similarities. n_pairs << n^2 keeps it cheap."""
    rng = np.random.default_rng(0)
    n = len(vectors)
    i = rng.integers(0, n, n_pairs)
    j = rng.integers(0, n, n_pairs)
    mask = i != j
    return (vectors[i[mask]] * vectors[j[mask]]).sum(axis=1)


fig, ax = plt.subplots(figsize=(9, 5))
for d in [2, 8, 64, 768]:
    v = random_unit_vectors(2000, d, seed=d)
    sims = pairwise_cosine_sample(v)
    sns.kdeplot(sims, ax=ax, label=f"d = {d}", linewidth=2)
ax.set_xlabel("cosine similarity")
ax.set_ylabel("density")
ax.set_title("Random unit vectors: pairwise cosine collapses to 0 as d grows")
ax.legend(title="dimension")
plt.tight_layout()
plt.savefig("../evals/figures/random_vectors_concentration.png", bbox_inches="tight")
plt.show()

Variance collapses approximately as $1/d$. By $d = 768$ — CLIP's projection dimension — the histogram is a narrow spike at zero. Anything that *isn't* near zero in this geometry is doing real work.

## Now real encoders: CLIP vs SigLIP-2

The interesting question: do trained encoders inherit this property, or do they distort it? The answer depends on the loss.

In [ ]:
encoders = {}
encoders["clip-vit-large"] = build_encoder(
    EmbeddingConfig(backend="clip", model_name="openai/clip-vit-large-patch14", batch_size=32)
)
encoders["siglip2-base"] = build_encoder(
    EmbeddingConfig(backend="siglip2", model_name="google/siglip2-base-patch16-256", batch_size=64)
)

In [ ]:
embeddings = {}
for name, enc in encoders.items():
    print(f"Encoding {N_SAMPLE} images with {name}...")
    embeddings[name] = enc.encode_images(sample_paths)
    print(f"  shape = {embeddings[name].shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, vecs in embeddings.items():
    sims = pairwise_cosine_sample(vecs, n_pairs=100_000)
    sns.kdeplot(sims, ax=ax, label=f"{name}  (μ = {sims.mean():.3f})", linewidth=2)
    print(f"{name:25s}  mean={sims.mean():+.4f}  std={sims.std():.4f}  median={np.median(sims):+.4f}")

# Reference line: where random unit vectors of the same dim would land
rand_vecs = random_unit_vectors(2000, embeddings["clip-vit-large"].shape[1], seed=99)
rand_sims = pairwise_cosine_sample(rand_vecs)
sns.kdeplot(rand_sims, ax=ax, label="random unit vectors (reference)", linestyle="--", linewidth=1.5, color="gray")

ax.axvline(0, color="black", linestyle=":", linewidth=1)
ax.set_xlabel("cosine similarity")
ax.set_ylabel("density")
ax.set_title("Pairwise cosine similarity: real encoders on 5,000 CelebA faces")
ax.legend()
plt.tight_layout()
plt.savefig("../evals/figures/encoder_anisotropy.png", bbox_inches="tight")
plt.show()

## What this picture shows

- **CLIP** (orange) is anisotropic. The mean pairwise similarity sits well above zero — most face pairs in CelebA look "kind of similar" to CLIP. The available dynamic range for distinguishing near-matches is small.
- **SigLIP-2 base** (blue) sits much closer to the reference line. The distribution is tighter and centered nearer zero.
- **Reference** (gray dashed): where random unit vectors would land. SigLIP-2 isn't fully isotropic — no trained encoder is — but it's noticeably closer to the geometric ideal.

**Why this matters for retrieval:** a similarity score of, say, 0.30 means very different things in these two spaces. In CLIP it's near the mode — barely above background. In SigLIP-2 it's far into the tail — strong evidence of semantic similarity.

This is the geometric reason SigLIP-2 retrieves better at every K. The tighter zero-centered distribution gives more bits of information per cosine score.

## A second view: nearest-neighbor distance contrast

An alternative diagnostic. For each query embedding, look at the ratio of its **furthest** to **nearest** neighbor cosine. If the encoder is doing useful work, this ratio should be far above 1 — there should be meaningful spread between "this is similar" and "this is not."

In [ ]:
def nn_contrast(vecs: np.ndarray, n_queries: int = 200, k: int = 100) -> np.ndarray:
    rng = np.random.default_rng(0)
    q_idx = rng.choice(len(vecs), size=n_queries, replace=False)
    sims_to_all = vecs[q_idx] @ vecs.T  # (n_queries, n_corpus)
    # Mask self
    for i, qi in enumerate(q_idx):
        sims_to_all[i, qi] = -np.inf
    top = np.partition(sims_to_all, -k, axis=1)[:, -k:]
    bot = np.partition(sims_to_all, k, axis=1)[:, :k]
    # Distance contrast: 1 - sim, then ratio of max to min.
    nearest = top.max(axis=1)
    furthest = bot.min(axis=1)
    return (1 - furthest) / (1 - nearest + 1e-8)


for name, vecs in embeddings.items():
    contrasts = nn_contrast(vecs)
    print(f"{name:25s}  nn-distance-contrast: median={np.median(contrasts):.2f}  p95={np.percentile(contrasts, 95):.2f}")

Higher contrast = more useful spread. SigLIP-2's higher contrast says nearest neighbors really are *much* closer than far ones, which is exactly what you want when you're going to use cosine ordering for retrieval.

## Summary

Two encoders. Same data. Different lossses, different geometries:

- CLIP packs everything into a narrow cone (anisotropy). Discriminative power suffers.
- SigLIP-2 spreads embeddings more uniformly. The hypersphere is used as it should be.

The Recall@K numbers in [`evals/results.md`](../evals/results.md) are the downstream consequence. The geometry is upstream of the metric.